In [3]:
# ==============================
# Imports
# ==============================

import os
import cv2
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

In [4]:
# ==============================
# Paths
# ==============================

DATASETS = r"C:\AliCode\Datasets"
ONLINE_DATASET = "data_online_line_width_alpha"
ONLINE_DATASET_PATH = os.path.join(DATASETS, ONLINE_DATASET)

os.chdir(ONLINE_DATASET_PATH)

In [ ]:
# ==============================
# Tight Crop Utility
# ==============================

def tight_crop_img(img_path, thresh_val=254):
    img = cv2.imread(img_path)

    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, thresh_val, 255, cv2.THRESH_BINARY_INV)

    coords = cv2.findNonZero(thresh)
    if coords is None:
        return img

    x, y, w, h = cv2.boundingRect(coords)
    return img[y:y+h, x:x+w]

In [ ]:
# ==============================
# Load Main CSV Index
# ==============================

df_main = None
files = [6000]  # dataset size selector

for csv_file in os.listdir(ONLINE_DATASET_PATH):
    if csv_file.startswith("main"):
        num = int(csv_file.split("_")[1].split(".")[0])
        if num in files:
            tmp = pd.read_csv(os.path.join(ONLINE_DATASET_PATH, csv_file))
            df_main = tmp if df_main is None else pd.concat([df_main, tmp])

df_main.reset_index(drop=True, inplace=True)

In [ ]:
# ==============================
# Fit MinMax Scaler (Train Only)
# ==============================

scaler = MinMaxScaler()

df_train_index = pd.read_csv("train.csv")

for record in tqdm(df_train_index.itertuples(), desc="Fitting scaler"):
    csv = pd.read_csv(record.csv)
    scaler.partial_fit(csv)

In [ ]:
# ==============================
# Feature-Based Image Generation
# ==============================

FEATURE_COLS = ['Pressure', 'X tilt', 'Y tilt', 'Height', 'Thickness']
POINT_SIZE = 0.08
DPI = 300

for record in tqdm(df_main.itertuples(), desc="Generating Images"):
    
    csv = pd.read_csv(record.csv)
    csv = csv[csv['Pressure'] > 0]

    if len(csv) == 0:
        continue

    # Scale
    scaled = scaler.transform(csv)
    csv_scaled = pd.DataFrame(scaled, columns=csv.columns)

    x = csv['X cood.']
    y = csv['Y cood.']

    for col in FEATURE_COLS:

        plt.clf()
        ax = plt.gca()
        ax.set_aspect('equal')
        ax.invert_yaxis()

        plt.scatter(
            x,
            y,
            s=POINT_SIZE,
            color='black',
            linewidth=0,
            alpha=csv_scaled[col]
        )

        plt.axis('off')

        name_parts = record.csv.split('/')
        name_parts[2] = f'img_{col}'
        os.makedirs('/'.join(name_parts[:-1]), exist_ok=True)

        out_path = '/'.join(name_parts)[:-4] + '.png'
        out_path = out_path.replace('csv', 'img')

        plt.savefig(out_path, dpi=DPI, bbox_inches='tight', pad_inches=0)
        plt.close()

        cropped = tight_crop_img(out_path, thresh_val=254)
        if cropped is not None:
            cv2.imwrite(out_path, cropped)

In [ ]:
# ==============================
# Generate Extended CSV (Leakproof)
# ==============================

files_in = [
    "train_leakproof_raw.csv",
    "val_leakproof_raw.csv",
    "test_leakproof_raw.csv"
]

files_out = [
    "train_leakproof.csv",
    "val_leakproof.csv",
    "test_leakproof.csv"
]

for file_in, file_out in zip(files_in, files_out):

    df = pd.read_csv(file_in)

    df["img_pressure"] = df["img"].apply(lambda x: x.replace('img', 'img_Pressure', 1))
    df["img_x_tilt"] = df["img"].apply(lambda x: x.replace('img', 'img_X tilt', 1))
    df["img_y_tilt"] = df["img"].apply(lambda x: x.replace('img', 'img_Y tilt', 1))
    df["img_height"] = df["img"].apply(lambda x: x.replace('img', 'img_Height', 1))
    df["img_thickness"] = df["img"].apply(lambda x: x.replace('img', 'img_Thickness', 1))

    df.to_csv(file_out, index=False)

In [5]:
# ==============================
# Generate Extended CSV (Leakproof - Dynamic)
# ==============================

import os
import pandas as pd

files_in = [
    "train_leakproof_raw2.csv",
    "val_leakproof_raw2.csv",
    "test_leakproof_raw2.csv"
]

files_out = [
    "train_leakproof2.csv",
    "val_leakproof2.csv",
    "test_leakproof2.csv"
]

# Detect all img_* feature folders automatically
feature_dirs = [
    d for d in os.listdir("C:\\AliCode\\Datasets\\data_online_line_width_alpha\\Dataset\\Data_1000")
    if os.path.isdir(os.path.join("C:\\AliCode\\Datasets\\data_online_line_width_alpha\\Dataset\\Data_1000", d)) and d.startswith("img_")
]

print("Detected features:", feature_dirs)
created_cols = [d.lower().replace(" ", "_") for d in feature_dirs]

for file_in, file_out in zip(files_in, files_out):

    df = pd.read_csv(file_in)

    for folder in feature_dirs:

        # Clean column name (lowercase + replace spaces)
        col_name = folder.lower().replace(" ", "_")

        df[col_name] = df["img"].apply(
            lambda x: x.replace("img", folder, 1)
        )

    df.to_csv(file_out, index=False)

print("Created columns:", created_cols)
print("Extended CSV files generated successfully.")

Detected features: ['img_acceleration', 'img_cos_theta', 'img_curvature', 'img_dt', 'img_dtheta', 'img_dvx', 'img_dvy', 'img_dx', 'img_dy', 'img_Pressure', 'img_sin_theta', 'img_speed', 'img_Stroke', 'img_stroke_duration', 'img_stroke_id', 'img_stroke_time', 'img_stroke_time_norm', 'img_theta', 'img_time_norm', 'img_vx', 'img_vy', 'img_X tilt', 'img_Y tilt']
Created columns: ['img_acceleration', 'img_cos_theta', 'img_curvature', 'img_dt', 'img_dtheta', 'img_dvx', 'img_dvy', 'img_dx', 'img_dy', 'img_pressure', 'img_sin_theta', 'img_speed', 'img_stroke', 'img_stroke_duration', 'img_stroke_id', 'img_stroke_time', 'img_stroke_time_norm', 'img_theta', 'img_time_norm', 'img_vx', 'img_vy', 'img_x_tilt', 'img_y_tilt']
Extended CSV files generated successfully.
